# 02 – RFM Customer Segmentation (K-Means)
Goal: group customers by Recency, Frequency, Monetary value and describe each group in business terms.

## 0. Setup & config
**Edit only this cell** to match your tables. Defaults follow the usual AdventureWorks CSV schema
(Sales_2015/16/17, Customers, Products, Territories). If you read from your Gold layer in Databricks,
set `USE_SPARK = True` and fix `GOLD_PATH` and the table names.

In [ ]:
import os, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None); pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
sns.set_theme(style="whitegrid")

USE_SPARK  = False                 # True inside Databricks (reads Delta from the Gold layer)
DATA_DIR   = "data"                # local folder with CSV exports (used when USE_SPARK = False)
GOLD_PATH  = "/mnt/gold"           # Databricks mount point of your gold container

SALES_TABLES    = ["Sales_2015", "Sales_2016", "Sales_2017"]   # or ["fact_sales"] if you have one Gold fact table
CUSTOMERS_TABLE = "Customers"
PRODUCTS_TABLE  = "Products"
TERRITORY_TABLE = "Territories"    # optional; set to None to skip

C = dict(order_date="OrderDate", order_no="OrderNumber", product="ProductKey", customer="CustomerKey",
         territory="TerritoryKey", qty="OrderQuantity", price="ProductPrice", cost="ProductCost")

for d in ("figures", "outputs", "audit"):
    os.makedirs(d, exist_ok=True)

def load(name):
    if USE_SPARK:
        return spark.read.format("delta").load(f"{GOLD_PATH}/{name}").toPandas()   # noqa: F821 (spark exists in Databricks)
    return pd.read_csv(f"{DATA_DIR}/{name}.csv")

## 1. Load tables and build one clean order-line table

In [ ]:
sales     = pd.concat([load(t) for t in SALES_TABLES], ignore_index=True)
customers = load(CUSTOMERS_TABLE)
products  = load(PRODUCTS_TABLE)
raw_tables = {"sales": sales, "customers": customers, "products": products}

## 2. Clean order lines (same rules as notebook 01)

In [ ]:
df = sales.drop_duplicates().dropna(subset=[C["customer"], C["order_date"]]).copy()
df[C["order_date"]] = pd.to_datetime(df[C["order_date"]])
df[C["customer"]] = df[C["customer"]].astype(int)
df = df[df[C["qty"]] > 0].merge(products[[C["product"], C["price"]]], on=C["product"], how="left")
df["revenue"] = df[C["qty"]] * df[C["price"]]
orders = df.groupby([C["customer"], C["order_no"]], as_index=False).agg(date=(C["order_date"], "max"), revenue=("revenue", "sum"))
print(orders.shape)

## 3. Build RFM features

In [ ]:
snapshot = orders["date"].max() + pd.Timedelta(days=1)
rfm = orders.groupby(C["customer"]).agg(
    recency=("date", lambda s: (snapshot - s.max()).days),
    frequency=(C["order_no"], "nunique"),
    monetary=("revenue", "sum"),
).reset_index()
rfm.describe()

In [ ]:
# RFM is heavily right-skewed -> log transform, then scale (K-Means uses distances, so scaling is mandatory)
from sklearn.preprocessing import StandardScaler
X = rfm[["recency", "frequency", "monetary"]].copy()
X["frequency"] = np.log1p(X["frequency"]); X["monetary"] = np.log1p(X["monetary"]); X["recency"] = np.log1p(X["recency"])
Xs = StandardScaler().fit_transform(X)

## 4. Choose k (elbow + silhouette)

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
ks, inertia, sil = range(2, 9), [], []
for k in ks:
    km = KMeans(n_clusters=k, n_init=10, random_state=42).fit(Xs)
    inertia.append(km.inertia_); sil.append(silhouette_score(Xs, km.labels_))
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(list(ks), inertia, marker="o"); ax[0].set_title("Elbow (inertia)"); ax[0].set_xlabel("k")
ax[1].plot(list(ks), sil, marker="o", color="green"); ax[1].set_title("Silhouette score"); ax[1].set_xlabel("k")
plt.tight_layout(); plt.savefig("figures/06_choose_k.png", dpi=150); plt.show()
print(dict(zip(ks, np.round(sil, 3))))

In [ ]:
K = 4   # <- set from the plots above (balance silhouette with interpretability); explain this choice in interviews
km = KMeans(n_clusters=K, n_init=10, random_state=42).fit(Xs)
rfm["segment_id"] = km.labels_

## 5. Profile and name the segments

In [ ]:
profile = rfm.groupby("segment_id").agg(customers=(C["customer"], "count"), recency=("recency", "mean"),
                                       frequency=("frequency", "mean"), monetary=("monetary", "mean"),
                                       total_revenue=("monetary", "sum"))
profile["customer_%"] = profile.customers / profile.customers.sum() * 100
profile["revenue_%"] = profile.total_revenue / profile.total_revenue.sum() * 100
# value score: high frequency + high monetary + LOW recency => better customer
score = (profile.frequency.rank() + profile.monetary.rank() + (-profile.recency).rank())
profile["value_rank"] = score.rank(ascending=False).astype(int)
profile.sort_values("value_rank")

In [ ]:
# Rule-based names: compare each segment with the overall average customer. Always sanity-check against the table above
# and edit the names by hand if they don't fit your data (a "Champion" must be RECENT, not just a big spender).
ref = rfm[["recency", "frequency", "monetary"]].mean()
def name_segment(r):
    recent, frequent, valuable = r.recency < ref.recency, r.frequency > ref.frequency, r.monetary > ref.monetary
    if recent and (frequent or valuable): return "Champions / Loyal"
    if recent:                            return "New / Recent low-spend"
    if frequent or valuable:              return "At-Risk high-value"
    return "Lost / Low-value"
profile["segment_name"] = profile.apply(name_segment, axis=1)
dup = profile.segment_name.duplicated(keep=False)
profile.loc[dup, "segment_name"] += " (" + profile.index[dup].astype(str) + ")"       # keep names unique
rfm = rfm.merge(profile[["segment_name", "value_rank"]], left_on="segment_id", right_index=True)
profile.sort_values("value_rank")[["segment_name", "customers", "customer_%", "recency", "frequency", "monetary", "revenue_%"]]

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
sns.scatterplot(data=rfm, x="recency", y="monetary", hue="segment_name", alpha=.6, ax=ax[0]); ax[0].set_yscale("log"); ax[0].set_title("Segments: recency vs monetary")
p = profile.sort_values("value_rank"); ax[1].bar(p.segment_name, p["revenue_%"], color="teal"); ax[1].set_title("Revenue share by segment (%)")
plt.setp(ax[1].get_xticklabels(), rotation=20); plt.tight_layout(); plt.savefig("figures/07_segments.png", dpi=150); plt.show()

## 6. Save results (feed these back into your Gold layer)

In [ ]:
out = rfm[[C["customer"], "recency", "frequency", "monetary", "segment_id", "segment_name"]]
out.to_csv("outputs/customer_segments.csv", index=False)
out.head()

In [ ]:
# --- DATABRICKS ONLY: write segments back to the Gold container as Delta, then expose in Synapse ---
# sdf = spark.createDataFrame(out)
# sdf.write.format("delta").mode("overwrite").save("/mnt/gold/customer_segments")
#
# -- Synapse serverless SQL (run in Synapse; adjust data source / path names to yours):
# CREATE VIEW gold.customer_segments AS
# SELECT * FROM OPENROWSET(BULK 'https://<storageaccount>.dfs.core.windows.net/gold/customer_segments/',
#                          FORMAT = 'DELTA') AS r;